# Milestone 5 — Single-case VLM tumor-size estimation (KiPA22 case 0)

> **INVALID RESULTS.** The saved outputs of this notebook came from the old blank (saturated-white) input images, caused by a CT intensity-preprocessing bug. They are preserved unchanged for transparency and must not be cited. The export scripts have since been fixed; see `docs/PHASE0_AUDIT.md`.

Model: `google/medgemma-1.5-4b-it` (gated, requires accepted HF terms + token).

This notebook:
1. Installs dependencies and logs in to Hugging Face (token via Colab secret, never hardcoded).
2. Loads the model in **bf16** (with a documented 4-bit fallback cell if bf16 runs out of memory).
3. Runs **Experiment A** (no pixel spacing) and **Experiment B** (with pixel spacing 0.585938 mm) on the *same* uploaded slice image.
4. Saves the full visible model responses, parses `major_mm`/`minor_mm`, and compares both against the reference measurement (major=16.33 mm, minor=13.98 mm) from our own MedVision-style pipeline.

No fine-tuning, no training, no other cases — case 0 only.

**Run cells top to bottom, in order.**

## Step 1 — Enable GPU
Before running anything: **Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU -> Save.**

In [ ]:
!nvidia-smi

## Step 2 — Install dependencies

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes huggingface_hub pillow

## Step 3 — Log in to Hugging Face (secure, no hardcoded token)

This reads your token from **Colab Secrets** (the key icon in the left sidebar), never from code in this notebook.
You must first: (a) accept the MedGemma terms on the model page, (b) create a *read* token in your HF account, (c) add it as a Colab secret named `HF_TOKEN` with notebook access enabled.
See the chat instructions for exact steps.

In [ ]:
from huggingface_hub import login

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    print("Loaded token from Colab secret 'HF_TOKEN'.")
except Exception:
    # Fallback: interactive, hidden-input prompt (never printed, never saved to the notebook file)
    from getpass import getpass
    hf_token = getpass("Enter your Hugging Face token: ")

login(token=hf_token)
hf_token = None  # drop the variable once logged in

## Step 4 — Upload the windowed CT slice
Upload `outputs/vlm_case0/slice_windowed.png` from your local repo (generated by `src/prepare_vlm_input.py`).

In [ ]:
from google.colab import files
from PIL import Image

uploaded = files.upload()  # select slice_windowed.png when prompted
image_filename = next(iter(uploaded.keys()))
ct_image = Image.open(image_filename).convert("RGB")
ct_image

## Step 5 — Load the model (bf16 first)

Try this cell first. If you hit a CUDA out-of-memory error, **skip this cell on re-run and instead run the "4-bit fallback" cell below it**, then continue with the rest of the notebook as normal — every later cell uses the same `model`/`processor` variables regardless of which loading path you used.

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "google/medgemma-1.5-4b-it"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print("Loaded in bf16.")

### 4-bit fallback (only run this if the bf16 cell above raised an out-of-memory error)

In [ ]:
# Only run this cell if Step 5 (bf16) failed with a CUDA out-of-memory error.
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MODEL_ID = "google/medgemma-1.5-4b-it"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
)
print("Loaded in 4-bit (nf4).")

## Step 6 — Define the two prompts and the reference measurement
No hidden chain-of-thought is requested; the full visible response is saved either way, but scoring only uses the parsed `major_mm`/`minor_mm` fields.

In [ ]:
REFERENCE_MAJOR_MM = 16.33
REFERENCE_MINOR_MM = 13.98

PROMPT_A = (
    "You are shown an axial CT slice of a kidney. Identify the kidney tumor (if visible) and estimate its size.\n"
    "Report the length of the tumor's longest axis (major axis) and the length of its shortest perpendicular axis (minor axis), in millimeters.\n"
    "Respond only in this exact format:\n"
    '<answer>{"major_mm": X, "minor_mm": Y}</answer>\n'
    "where X and Y are numbers in millimeters."
)

PROMPT_B = (
    "You are shown an axial CT slice of a kidney. This image has a pixel spacing of 0.585938 mm x 0.585938 mm per pixel\n"
    "(each pixel represents 0.585938 mm of real-world distance, in both the horizontal and vertical directions).\n"
    "Using this scale, identify the kidney tumor (if visible) and estimate its size.\n"
    "Report the length of the tumor's longest axis (major axis) and the length of its shortest perpendicular axis (minor axis), in millimeters.\n"
    "Respond only in this exact format:\n"
    '<answer>{"major_mm": X, "minor_mm": Y}</answer>\n'
    "where X and Y are numbers in millimeters."
)

print(PROMPT_A)
print()
print(PROMPT_B)

## Step 7 — Inference helper

In [ ]:
def run_prompt(image, prompt_text, max_new_tokens=300):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt_text},
            ],
        }
    ]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device, dtype=model.dtype if hasattr(model, "dtype") else torch.bfloat16)

    input_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        generation = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        generation = generation[0][input_len:]

    return processor.decode(generation, skip_special_tokens=True)

## Step 8 — Run Experiment A (no spacing) and Experiment B (with spacing)

In [ ]:
response_a = run_prompt(ct_image, PROMPT_A)
print("=== Experiment A raw response ===")
print(response_a)

In [ ]:
response_b = run_prompt(ct_image, PROMPT_B)
print("=== Experiment B raw response ===")
print(response_b)

## Step 9 — Save raw responses, parse structured output, compare to reference

In [ ]:
import re
import json
import csv

def parse_answer(raw_text):
    """Extract the <answer>{...}</answer> JSON block. Returns (major_mm, minor_mm) or (None, None) on failure."""
    match = re.search(r"<answer>\s*(\{.*?\})\s*</answer>", raw_text, re.DOTALL)
    if not match:
        return None, None
    try:
        parsed = json.loads(match.group(1))
        return parsed.get("major_mm"), parsed.get("minor_mm")
    except json.JSONDecodeError:
        return None, None

major_a, minor_a = parse_answer(response_a)
major_b, minor_b = parse_answer(response_b)

def err(pred, ref):
    if pred is None:
        return None, None
    abs_error = abs(pred - ref)
    pct_error = abs_error / ref * 100.0
    return round(abs_error, 3), round(pct_error, 2)

rows = []
for exp_name, major_pred, minor_pred, raw in [
    ("A_no_spacing", major_a, minor_a, response_a),
    ("B_with_spacing", major_b, minor_b, response_b),
]:
    major_abs_err, major_pct_err = err(major_pred, REFERENCE_MAJOR_MM)
    minor_abs_err, minor_pct_err = err(minor_pred, REFERENCE_MINOR_MM)
    rows.append({
        "experiment": exp_name,
        "major_mm_pred": major_pred,
        "minor_mm_pred": minor_pred,
        "major_mm_ref": REFERENCE_MAJOR_MM,
        "minor_mm_ref": REFERENCE_MINOR_MM,
        "major_abs_error_mm": major_abs_err,
        "major_pct_error": major_pct_err,
        "minor_abs_error_mm": minor_abs_err,
        "minor_pct_error": minor_pct_err,
    })

for r in rows:
    print(r)

In [ ]:
# Save raw responses (full transparency) and the results CSV, then download both to your Mac.
with open("raw_responses.txt", "w") as f:
    f.write("=== Experiment A (no spacing) — prompt ===\n" + PROMPT_A + "\n\n")
    f.write("=== Experiment A — raw response ===\n" + response_a + "\n\n")
    f.write("=== Experiment B (with spacing) — prompt ===\n" + PROMPT_B + "\n\n")
    f.write("=== Experiment B — raw response ===\n" + response_b + "\n")

fieldnames = list(rows[0].keys())
with open("vlm_case0_results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print("Saved raw_responses.txt and vlm_case0_results.csv")
files.download("raw_responses.txt")
files.download("vlm_case0_results.csv")

## Step 10 — After downloading
Move the two downloaded files into your local repo at:
- `outputs/vlm_case0/raw_responses.txt`
- `results/vlm_case0_results.csv`

so they sit alongside `outputs/vlm_case0/slice_windowed.png`.